In [2]:
from joblib import Parallel, delayed
from tqdm.notebook import tqdm

import sys
import pickle
import numpy as np
import pandas as pd
import networkx as nx
import scipy
from itertools import combinations

sys.path.insert(0, "src")

from grn import grn


In [ ]:
import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.optim as optim
import time
import copy
from itertools import combinations
from joblib import Parallel, delayed
from tqdm.notebook import tqdm
from sklearn.linear_model import LinearRegression

# ============================================================
# INITIALIZE GLOBAL STORAGE
# ============================================================
all_results = []
higher_order_results = [] # New list to store our high-order evaluations

for network_seed in range(10):
    np.random.seed(network_seed)

    # manually adjust GRN hyperparameters
    G = grn().add_structure(
        n=256,
        k=1,
        alpha=1e-99,
        beta=0.99,       # 1 - 1/r
        gamma=0.01,      # 1/r
        kappa=10,        # --w
        delta_in=100,
        delta_out=100,
    )

    G.simulate_rna(
        s=1e-4,
        dt=1e-2,
        tmax=20000,
        burnin=5000,
        tol=1e-3,
        save=True,
    )

    x_wt = np.zeros(256)
    y_wt = G.rna

    single_ko_expression = np.zeros((G.n, G.n))
    single_ko_logfc = np.zeros((G.n, G.n))
    single_ko_convergence = {}

    for gene in tqdm(range(G.n), desc=f"Network {network_seed}: single-gene KOs"):
        result = G.ko_nodes(
            genes=[gene],
            stats=("new_rna", "logfc", "convergence"),
            s=1e-4,
            dt=1e-2,
            tmax=20000,
            burnin=5000,
            tol=1e-3,
        )

        single_ko_expression[gene, :] = result["new_rna"].flatten()
        single_ko_logfc[gene, :] = result["logfc"].flatten()
        single_ko_convergence[gene] = result["convergence"]

    x_single_ko = np.eye(G.n)
    y_single_ko = single_ko_expression
    
    # ------------------------------------------------------------
    # DOUBLE KO SIMULATION
    # ------------------------------------------------------------
    pairs = list(combinations(range(G.n), 2))

    def run_double_ko(pair):
        a, b = pair
        result = G.ko_nodes(
            genes=[a, b],
            stats=("new_rna", "logfc", "convergence"),
            s=1e-4, dt=1e-2, tmax=20000, burnin=5000, tol=1e-3,
        )
        return (
            np.asarray(result["new_rna"]).flatten(),
            np.asarray(result["logfc"]).flatten(),
            np.asarray(result["convergence"]),
        )

    results = Parallel(n_jobs=64, pre_dispatch="n_jobs", prefer="processes", verbose=0)(
        delayed(run_double_ko)(pair) for pair in pairs
    )

    double_expression = np.asarray([r[0] for r in results], dtype=np.float32)

    # prepare as training data
    pairs = np.array(pairs, dtype=np.int32)
    x_double_ko = np.zeros((pairs.shape[0], G.n))
    rows = np.arange(pairs.shape[0])
    x_double_ko[rows, pairs[:, 0]] = 1.0
    x_double_ko[rows, pairs[:, 1]] = 1.0
    y_double_ko = double_expression

    # ------------------------------------------------------------
    # HIGHER ORDER KO SIMULATION (Sets 3 to 10)
    # ------------------------------------------------------------
    select_size = 200
    higher_expression = {}
    all_select_sets = []
    X_list = []

    for set_size in tqdm(range(3, 11), desc=f"Network {network_seed}: higher-order KOs"):
        select_sets = set()
        while len(select_sets) < select_size:
            comb = tuple(sorted(np.random.choice(G.n, size=set_size, replace=False)))
            select_sets.add(comb)
        select_sets = list(select_sets)

        def run_combination_ko(comb):
            res = G.ko_nodes(genes=list(comb), stats=("new_rna",), s=1e-4, dt=1e-2, tmax=20000, burnin=5000, tol=1e-3)
            return np.asarray(res["new_rna"]).flatten()

        results_ho = Parallel(n_jobs=64, prefer="processes")(
            delayed(run_combination_ko)(combination) for combination in select_sets
        )

        higher_expression[set_size] = np.asarray(results_ho, dtype=np.float32)
        
        select_sets_arr = np.array(select_sets, dtype=np.int32)
        all_select_sets.append(select_sets_arr)

        X_subset = np.zeros((select_size, G.n), dtype=np.float32)
        rows_ho = np.arange(select_size)[:, None]
        X_subset[rows_ho, select_sets_arr] = 1.0
        X_list.append(X_subset)


    # ============================================================
    # 3 TRAINING RUNS ON THIS GRN
    # ============================================================
    for training_seed in range(3):
        torch.manual_seed(training_seed)
        device = torch.device("cpu")

        model = nn.Sequential(
            nn.Linear(256, 512), nn.GELU(),
            nn.Linear(512, 512), nn.GELU(),
            nn.Linear(512, 256)
        ).to(device)

        optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)

        # process data
        rng = np.random.default_rng(training_seed)
        n_pairs = pairs.shape[0]
        nt, nv = int(0.5 * n_pairs), int(0.1 * n_pairs)
        indices = rng.permutation(n_pairs)
        
        test_idx = indices[:nt]
        val_idx = indices[nt:nt + nv]
        train_idx = indices[nt + nv:]

        xtr = np.concatenate([x_wt[None, :], x_single_ko, x_double_ko[train_idx]], axis=0)
        ytr = np.concatenate([y_wt[None, :], y_single_ko, y_double_ko[train_idx]], axis=0)
        xv = x_double_ko[val_idx]
        yv = y_double_ko[val_idx]
        x_test = x_double_ko[test_idx]
        y_test = y_double_ko[test_idx]

        mean = np.mean(ytr, axis=0)
        scale = np.maximum(ytr.std(axis=0), 1e-6)
        s_ytr = (ytr - mean) / scale

        xtr_t = torch.as_tensor(xtr, dtype=torch.float32, device=device)
        s_ytr_t = torch.as_tensor(s_ytr, dtype=torch.float32, device=device)
        xv_t = torch.as_tensor(xv, dtype=torch.float32, device=device)

        # Training loop
        max_epochs = 3000
        batch_size = 256
        lr = 1e-3
        patience = 40
        min_lr = 1e-5
        lr_patience = 12

        best, best_epoch, stale, lr_stale = np.inf, 0, 0, 0
        best_model_weights = None

        for epoch in range(1, max_epochs + 1):
            model.train()
            permutation = torch.randperm(xtr_t.shape[0], device=device)
            for start in range(0, xtr_t.shape[0], batch_size):
                ii = permutation[start:(start + batch_size)]
                optimizer.zero_grad(set_to_none=True)
                loss = torch.mean((model(xtr_t[ii]) - s_ytr_t[ii]) ** 2)
                loss.backward()
                optimizer.step()

            model.eval()
            with torch.inference_mode():
                val_scaled = torch.cat([model(chunk) for chunk in xv_t.split(1024)]).cpu().numpy()
            
            val_pred = val_scaled.astype(np.float32) * scale + mean
            raw_mse = float(np.mean((val_pred - yv) ** 2))

            if raw_mse < best:
                best = raw_mse
                stale = 0
                lr_stale = 0
                best_model_weights = copy.deepcopy(model.state_dict())
            else:
                stale += 1
                lr_stale += 1

            if lr_stale >= lr_patience and lr > min_lr:
                lr = max(min_lr, lr * .5)
                for group in optimizer.param_groups:
                    group["lr"] = lr
                lr_stale = 0

            if stale >= patience:
                break

        if best_model_weights is not None:
            model.load_state_dict(best_model_weights)

        # Fit Baseline Linear Model
        linear_model = LinearRegression(fit_intercept=True)
        linear_model.fit(xtr.astype(np.float64), s_ytr.astype(np.float64))

        # ------------------------------------------------------------
        # EVALUATE ON HIGHER ORDER DATASETS
        # ------------------------------------------------------------
        model.eval()
        for i, set_size in enumerate(range(3, 11)):
            X_subset = X_list[i]
            select_sets_arr = all_select_sets[i]
            y_true = higher_expression[set_size]
            
            # Additive Baseline for k-gene KO
            additive = y_single_ko[select_sets_arr].sum(axis=1) - (set_size - 1) * y_wt
            
            # Linear model eval
            lin_scaled = linear_model.predict(X_subset.astype(np.float64))
            lin_pred = lin_scaled.astype(np.float32) * scale + mean
            
            # MLP eval
            X_subset_t = torch.as_tensor(X_subset, dtype=torch.float32, device=device)
            with torch.inference_mode():
                mlp_scaled = torch.cat([model(chunk) for chunk in X_subset_t.split(1024)]).cpu().numpy()
            mlp_pred = mlp_scaled.astype(np.float32) * scale + mean
            
            # Calculate MSE
            add_mse = float(np.mean((additive - y_true) ** 2))
            lin_mse = float(np.mean((lin_pred - y_true) ** 2))
            mlp_mse = float(np.mean((mlp_pred - y_true) ** 2))
            
            # Calc percentage improvement over additive
            lin_improv = 100 * (1 - lin_mse / add_mse)
            mlp_improv = 100 * (1 - mlp_mse / add_mse)
            
            higher_order_results.append({
                "Network Seed": network_seed,
                "Training Seed": training_seed,
                "Combination Size (k)": set_size,
                "Linear Improv (%)": lin_improv,
                "MLP Improv (%)": mlp_improv
            })

# ================================================================
# FINAL RESULT TABLES
# ================================================================

# Convert higher order metrics into DataFrame
ho_df = pd.DataFrame(higher_order_results)

# Helper function to format triplicate values as a list string
def format_triplicate(series):
    return "[" + ", ".join([f"{val:.1f}%" for val in series.values]) + "]"

# Group by Combination Size & Network Seed, applying the formatting function to aggregate 3 training runs
ho_aggregated = ho_df.groupby(["Combination Size (k)", "Network Seed"]).agg({
    "Linear Improv (%)": format_triplicate,
    "MLP Improv (%)": format_triplicate
}).reset_index()

# Pivot for Final Display
pivot_lin = ho_aggregated.pivot(index="Combination Size (k)", columns="Network Seed", values="Linear Improv (%)")
pivot_mlp = ho_aggregated.pivot(index="Combination Size (k)", columns="Network Seed", values="MLP Improv (%)")

pivot_lin.columns.name = "GRN Random Seed"
pivot_mlp.columns.name = "GRN Random Seed"

print("\n=== Linear Model: MSE Improvement over Additive Baseline (Triplicates) ===")
display(pivot_lin)

print("\n=== MLP Model: MSE Improvement over Additive Baseline (Triplicates) ===")
display(pivot_mlp)

Network 0: single-gene KOs:   0%|          | 0/256 [00:00<?, ?it/s]